In [1]:
from pyspark.sql.functions import to_date, count, current_date, current_timestamp, col
from delta.tables import DeltaTable

# Enable schema evolution for MERGE operations
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ----------------------------------------------------------------------
# Read Bronze and compute one row per day
# ----------------------------------------------------------------------
df = spark.read.table("top_headlines")

available_cols = df.columns
date_source = "ingested_at" if "ingested_at" in available_cols else "last_seen_at"
print(f"Computing daily summary from column: {date_source}")

summary_today = (
    df.withColumn("day", to_date(col(date_source)))
      .groupBy("day")
      .agg(count("*").alias("articles_ingested"))
      .withColumn("last_updated", current_timestamp())
)

print(f"Days computed from current Bronze: {summary_today.count()}")
summary_today.orderBy("day", ascending=False).show(10, truncate=False)

# ----------------------------------------------------------------------
# MERGE into daily_ingestion_summary — preserves historical days
# ----------------------------------------------------------------------
table_name = "daily_ingestion_summary"

if spark.catalog.tableExists(table_name):
    try:
        delta_tbl = DeltaTable.forName(spark, table_name)
        print("✅ Table exists as Delta, performing MERGE with schema evolution")

        (delta_tbl.alias("t")
            .merge(summary_today.alias("s"), "t.day = s.day")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        print("✅ Merge complete — existing days updated, new days inserted, schema evolved")

    except Exception as e:
        print(f"⚠️ Couldn't open as Delta ({e}) — falling back to overwrite with schema change")
        (summary_today.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name))
        print("✅ Table overwritten with new schema (one-time recovery)")
else:
    print("📦 Table doesn't exist — creating it for the first time")
    summary_today.write.saveAsTable(table_name)
    print("✅ Table created")

# ----------------------------------------------------------------------
# Verify
# ----------------------------------------------------------------------
final = spark.read.table(table_name)
print(f"\nTotal days tracked: {final.count()}")
print(f"Columns: {final.columns}")
final.orderBy("day", ascending=False).show(10, truncate=False)

StatementMeta(, 2ac3e0d8-2500-4953-95d0-25b222e296e1, 3, Finished, Available, Finished, False)

Computing daily summary from column: ingested_at
Days computed from current Bronze: 1
+----------+-----------------+--------------------------+
|day       |articles_ingested|last_updated              |
+----------+-----------------+--------------------------+
|2026-05-13|18               |2026-05-13 09:06:19.207729|
+----------+-----------------+--------------------------+

✅ Table exists as Delta, performing MERGE with schema evolution
✅ Merge complete — existing days updated, new days inserted, schema evolved

Total days tracked: 2
Columns: ['day', 'articles_ingested', 'last_updated']
+----------+-----------------+--------------------------+
|day       |articles_ingested|last_updated              |
+----------+-----------------+--------------------------+
|2026-05-13|18               |2026-05-13 09:06:21.725015|
|2026-05-12|20               |2026-05-13 09:02:51.894963|
+----------+-----------------+--------------------------+

